# 3. RNN-T：二维格点、搜索与流式缓存

**研究问题**：声学时间 `t` 与已输出 token 数 `u` 如何共同决定概率？blank 与非 blank 怎样改变状态？分块处理时需要保留哪些信息？

**方法**：合成声学特征、微型 Encoder/Predictor/Joint 和可控评分函数，依次检查 Exp007–010。

```text
声学特征 [B,T,F] → Encoder h_t ───────────┐
历史 token → Prediction Network g_u ──────┤→ Joint [B,T,U+1,V]
                                            └→ 二维 RNNT Loss
推理 (t,u)：blank → (t+1,u)；token → (t,u+1)
流式 chunk：缓存 Encoder hidden state + 已输出 token
```

**边界**：小格点、随机模型与教学搜索；不测真实识别率、RTF 或端到端流式延迟。对应平台题目：`asr_rnnt_minimal`、`asr_rnnt_greedy`、`asr_rnnt_beam`、`asr_rnnt_streaming`。
**代码来源**：本仓库 `experiments/ctc_rnnt_aed/run.py`（基础版本提交 `e86b41b`）；Notebook 将其教学函数拆为逐格实验，不映射外部项目源码。


In [ ]:
import math
import torch
import torch.nn as nn


## Exp007：Mini-RNN-T 与前向 Loss

**原理**：Encoder 处理声学帧，Prediction Network 只处理非 blank 的历史 token，Joint 在每个 `(t,u)` 合并两路状态。训练时为目标长度 `U` 构造 `U+1` 个历史状态；前向格点将所有合法二维路径用 `logaddexp` 求和。

**形状**：声学输入 `[B,T,F]`，预测网络输入 `[B,U+1]`，Joint logits `[B,T,U+1,V]`；格点 `[T+1,U+1]`，终点标量 `-log P(Y|X)`。此例采用 BOS 与真实目标作为历史输入。

In [ ]:
class MiniRNNT(nn.Module):
    def __init__(self, acoustic_dim=3, vocab=4, dim=8):
        super().__init__()
        self.encoder = nn.Linear(acoustic_dim, dim)
        self.embedding = nn.Embedding(vocab, dim)
        self.predictor = nn.GRU(dim, dim, batch_first=True)
        self.joint = nn.Linear(dim, vocab)

    def forward(self, acoustic, history):
        encoded = self.encoder(acoustic)
        predicted, _ = self.predictor(self.embedding(history))
        return self.joint(torch.tanh(encoded[:, :, None, :] + predicted[:, None, :, :]))

def rnnt_forward(log_probs, target, blank=0):
    t_max, u_states, _ = log_probs.shape
    assert u_states == len(target) + 1
    lattice = [[log_probs.new_tensor(float('-inf')) for _ in range(u_states)] for _ in range(t_max + 1)]
    lattice[0][0] = log_probs.new_tensor(0.)
    for t in range(t_max):
        for u in range(u_states):
            lattice[t + 1][u] = torch.logaddexp(lattice[t + 1][u], lattice[t][u] + log_probs[t, u, blank])
            if u < len(target):
                lattice[t][u + 1] = torch.logaddexp(lattice[t][u + 1], lattice[t][u] + log_probs[t, u, target[u]])
    return lattice[-1][-1]


In [ ]:
torch.manual_seed(7)
model = MiniRNNT(acoustic_dim=3,vocab=4,dim=8)
features = torch.randn(1,2,3)
history = torch.tensor([[0,1]])
logits = model(features,history)
loss = -rnnt_forward(logits[0].log_softmax(-1),target=[1])
assert logits.shape == (1,2,2,4) and torch.isfinite(loss)
loss.backward()
assert model.joint.weight.grad is not None
print('Joint',tuple(logits.shape),'RNNT loss',loss.detach().item())
p = torch.tensor([[[.7,.3],[.4,.6]],[[.2,.8],[.5,.5]]])
exact = p[0,0,1]*p[0,1,0]*p[1,1,0] + p[0,0,0]*p[1,0,1]*p[1,1,0]
assert torch.allclose(rnnt_forward(p.log(),[1]).exp(),exact)
print('两条合法路径总概率',float(exact))

## Exp008：RNN-T Greedy Search

**原理**：blank 只推进声学帧，非 blank 追加 token 后仍停留在当前帧，因此同一帧可以输出多个 token。输出上限防止教学评分器永远不发 blank。

**形状**：评分器输入 `(t, 已输出 token 元组)`，返回 `[V]`；输出 `(token 元组, 是否因上限截断)`。

In [ ]:
def rnnt_greedy(score, frames, max_tokens, blank=0):
    tokens, t = (), 0
    while t < frames:
        token = int(torch.argmax(score(t, tokens)))
        if token == blank:
            t += 1
        else:
            tokens += (token,)
            if len(tokens) == max_tokens:
                return tokens, True
    return tokens, False


In [ ]:
score = lambda t,p: torch.tensor([0.,2.]) if t==0 and len(p)<2 else torch.tensor([3.,0.])
assert rnnt_greedy(score,frames=2,max_tokens=5) == ((1,1),False)
assert rnnt_greedy(lambda t,p: torch.tensor([0.,1.]),frames=2,max_tokens=2) == ((1,1),True)
print('同一帧重复输出：',rnnt_greedy(score,2,5))

## Exp009：RNN-T Beam Search

**原理**：同一声学帧中不断扩展非 blank 分支；blank 分支进入下一帧。不同二维路径到达相同 token 前缀时要合并概率质量。本实现以 `max_tokens` 有界枚举，每帧结束后按 `beam_size` 剪枝，是小词表教学搜索。

**形状**：评分器输入 `(t,prefix)` 返回 `[V]` log 概率；输出若干 `(prefix, log 概率)`。保留重复 token，不做 CTC Collapse。

In [ ]:
def rnnt_beam(score, frames, beam_size, max_tokens, blank=0):
    """Bounded small-example search, merging identical token prefixes at each frame."""
    beams = {(): 0.}
    for t in range(frames):
        pending = dict(beams)
        completed = {}
        for _ in range(max_tokens + 1):
            if not pending:
                break
            extensions = {}
            for prefix, mass in pending.items():
                probabilities = score(t, prefix).tolist()
                ended = mass + probabilities[blank]
                completed[prefix] = torch.logaddexp(torch.tensor(completed.get(prefix, -math.inf)), torch.tensor(ended)).item()
                if len(prefix) < max_tokens:
                    for token, value in enumerate(probabilities):
                        if token != blank:
                            child = prefix + (token,)
                            extensions[child] = torch.logaddexp(torch.tensor(extensions.get(child, -math.inf)), torch.tensor(mass + value)).item()
            pending = extensions
        order = sorted(completed, key=lambda p: (-completed[p], p))[:beam_size]
        beams = {p: completed[p] for p in order}
    return sorted(beams.items(), key=lambda x: (-x[1], x[0]))


In [ ]:
def beam_score(t,prefix):
    if not prefix: return torch.tensor([math.log(.5),math.log(.5)])
    return torch.tensor([math.log(.8),math.log(.2)])
ranked = rnnt_beam(beam_score,frames=2,beam_size=20,max_tokens=2)
merged = dict(ranked)
expected = .5*.8*.8 + .5*.5*.8  # 第 0 帧发 token；或先 blank 再发 token
assert abs(math.exp(merged[(1,)])-expected) < 1e-6
print('合并后的前缀分数：',ranked[:3])

## Exp010：Streaming RNN-T

**原理**：因果 GRU 的 hidden state 保存历史声学信息；已经输出的 token 元组保存 Prediction 侧历史。逐 chunk 处理时两者都不能重置。对同一编码帧发非 blank 后仍重复查询评分器，直到 blank 或每帧符号上限。

**形状**：输入 chunk `[B,C,F]`，Encoder 输出 `[B,C,D]`，cache `[1,B,D]`；解码输出为 token 元组。用相同权重比较整段与分块结果。

In [ ]:
class CausalEncoder(nn.Module):
    def __init__(self, feature_dim=3, hidden_dim=4):
        super().__init__()
        self.rnn = nn.GRU(feature_dim, hidden_dim, batch_first=True)

    def forward_chunk(self, chunk, cache=None):
        return self.rnn(chunk, cache)

def streaming_rnnt(chunks, encoder, score, max_tokens=8, max_symbols_per_frame=4, blank=0):
    cache, output = None, ()
    for chunk in chunks:
        encoded, cache = encoder.forward_chunk(chunk, cache)
        for t in range(encoded.shape[1]):
            emitted_on_frame = 0
            while emitted_on_frame < max_symbols_per_frame:
                token = int(torch.argmax(score(encoded[:, t], output)))
                if token == blank:
                    break
                output += (token,)
                emitted_on_frame += 1
                if len(output) >= max_tokens:
                    return output, cache, True
    return output, cache, False


In [ ]:
torch.manual_seed(7)
encoder = CausalEncoder(feature_dim=3,hidden_dim=4)
x = torch.randn(1,5,3)
def stream_score(frame,prefix):
    return torch.tensor([0.,2.]) if not prefix else torch.tensor([2.,0.])
full = streaming_rnnt([x],encoder,stream_score)
chunked = streaming_rnnt([x[:,:2],x[:,2:4],x[:,4:]],encoder,stream_score)
assert full[0] == chunked[0] and full[2] == chunked[2]
assert torch.allclose(full[1],chunked[1],atol=1e-6)
print('整段/分块 token',full[0],chunked[0],'cache',tuple(chunked[1].shape))

**验证依据与适用边界**：小格点概率与手算两路径一致、梯度有限、同权重分块与整段缓存一致。这不证明复杂 Encoder 的流式等价性，也没有音频前端、训练调度、真实语音评测或延迟测量。